> **Disclaimer.** This project is educational and is not investment advice. Backtest results are historical and do not predict future results.

# How many contracts, on how small an account, and against the CDI

The WIN optimized rule (20 days, bands x1.5, every 60 minutes from 12:00, exit 18:20, back-inside-the-band stop at every bar) on a small account: what sizing to use, what the risk is, and how it compares with leaving the money in the CDI while adding a fixed amount each month. Everything here is the **Python model** with costs (B3 fee R$ 0.25 plus one tick a side); the MetaTrader 5 runs it was checked against are in `docs/mql5-noise-area.md`.

**Read this first**
- **The rule was chosen on 2021-12 to 2024 and its 2025-26 result has been read once** (`winfut_final_test.ipynb`). This notebook is about sizing and risk, **not** about whether the edge is real, and it cannot select anything. Every "history" number here is in sample, which is why the risk simulations also run with **half the edge** and **none**.
- A P&L here is **per contract** and daily, from one run with one contract a day. Contracts, deposits and the stop-out only multiply it, so one run serves every section.
- Closing balances hide the swings inside a day. Section 4 adds the worst floating loss of each day.
- Taxes and the broker's fees beyond B3's are ignored, on the CDI side as on the day-trade side.

| Section | Question |
| --- | --- |
| 1 | The inputs: the per-contract P&L, the CDI, the worst floating loss of each day |
| 2 | What the paper's volatility sizing does on an account of R$ 1,000 to R$ 100,000 |
| 3 | One contract per R$ X of balance, with X from R$ 1,000 to R$ 10,000 |
| 4 | Risk of ruin, with a broker that stops you out at a 60% loss |
| 5 | When to add the second contract, and how long it takes |
| 6 | Head to head with the CDI, with a monthly deposit of R$ 250 to R$ 1,250 |


## 1. Inputs

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd

from b3core.benchmark import compare_with_cdi
from b3core.intraday_momentum import run_intraday_momentum, strategy_exposure
from b3core.intraday_search import make_config
from b3core.winfut import load_bars

pd.options.display.width = 220
pd.options.display.max_columns = 40
pd.options.display.max_rows = 200

PARAMS = {"lookback": 20, "band_mult": 1.5, "trade_freq": 60, "first_decision": 720, "exit_minute": 1100,
          "stop": "current_band", "stop_check": "every_bar", "sides": "both"}
SLIPPAGE_TICKS = 1.0  # per side, on top of the B3 fee (the search used 1)
SEED = 20261118

bars = pd.concat([load_bars("../samples"), load_bars("../samples", final=True)])  # 2021-12-01 to the end of the final period
cdi_raw = pd.read_csv("../samples/ifr2_universe/cdi.csv")
cdi = pd.Series(cdi_raw["rate"].to_numpy(dtype=float), index=pd.to_datetime(cdi_raw["date"], format="%d/%m/%Y")).sort_index()
END = min(cdi.index.max(), bars.index.max().normalize())
bars = bars[bars.index < END + pd.Timedelta(days=1)]  # the window both series cover
years = (bars.index[-1] - bars.index[0]).days / 365.25
print(f"window {bars.index[0]:%Y-%m-%d} to {bars.index[-1]:%Y-%m-%d} ({years:.2f} years), CDI up to {cdi.index.max():%Y-%m-%d}")

# one contract a day, with costs: the daily net P&L per contract
cfg1 = replace(make_config(PARAMS, "WIN", slippage_ticks=SLIPPAGE_TICKS), fixed_contracts=1, margin_per_contract=0.0)
res1 = run_intraday_momentum(bars, cfg1)
per_contract = res1["net"].fillna(0.0)
day_has_trade = per_contract != 0
print(f"per contract and day: mean R$ {per_contract.mean():.2f}, standard deviation R$ {per_contract.std():.1f}, "
      f"worst R$ {per_contract.min():.0f}, best R$ {per_contract.max():.0f}; a P&L on {day_has_trade.mean():.0%} of {len(per_contract):,} days")

**The worst floating loss of each day, per contract.** A broker's stop-out acts on the equity, which includes the open position. For each bar the position held was marked at the bar's worst price (the low for a long, the high for a short), on top of what the day had already made. It is an estimate from one-minute bars: a spike inside a minute is not in it.

In [ ]:
exposure = strategy_exposure(bars, cfg1).to_numpy(dtype=float)  # the position held DURING each bar
close, high, low = (bars[k].to_numpy(dtype=float) for k in ("close", "high", "low"))
day = pd.DatetimeIndex(bars.index).normalize()
first_bar = np.r_[True, day[1:] != day[:-1]]
previous = np.r_[close[0], close[:-1]]
previous[first_bar] = close[first_bar]  # nothing is carried over a night: a day starts flat
step = exposure * (close - previous)
made_before = pd.Series(step).groupby(day).cumsum().to_numpy() - step
worst_move = np.where(exposure > 0, exposure * (low - previous), np.where(exposure < 0, exposure * (high - previous), 0.0))
trough = (pd.Series((made_before + worst_move) * 0.2).groupby(day).min().clip(upper=0.0)).reindex(per_contract.index).fillna(0.0)
trough = np.minimum(trough, per_contract.clip(upper=0.0))  # never shallower than the day's own close
print(f"worst floating loss in one day, per contract: R$ {trough.min():.0f} on {trough.idxmin():%Y-%m-%d}; "
      f"days worse than R$ -300: {(trough < -300).sum()}")
print(f"the deepest run of days, per contract (closing P&L): R$ {(per_contract.cumsum() - per_contract.cumsum().cummax()).min():.0f}")

## 2. The paper's volatility sizing, on an account of R$ 1,000 to R$ 100,000

The paper sizes each day as `balance / (day open x point value) x min(target volatility / volatility, cap)`, rounded to whole contracts: 2% and a cap of 4 here. A WIN contract is worth about R$ 21,000 at an index of 105,000 and R$ 34,000 at 171,000, so a small account rounds to **zero** contracts. The table counts, for each balance, the days on which the rule wanted a position, how many were sized to zero, the average contracts on the rest, the leverage that implies (contracts x contract value / balance) and the result. The B3 margin of R$ 1,000 a contract is on; it never binds here.

In [ ]:
dopen = bars["open"].groupby(day).first()
rows = []
for capital in (1_000, 2_500, 5_000, 10_000, 25_000, 100_000):
    cfg = replace(make_config(PARAMS, "WIN", slippage_ticks=SLIPPAGE_TICKS, capital=float(capital)), margin_per_contract=1_000.0)
    r = run_intraday_momentum(bars, cfg)
    wanted = (exposure != 0)
    wanted_days = pd.Series(wanted).groupby(day).any().reindex(r.index).fillna(False)
    w = r[wanted_days & r["ret"].notna()]
    traded = w[w["contracts"] > 0]
    prev_aum = r["aum"].shift(1).fillna(capital)
    leverage = (traded["contracts"] * dopen.reindex(traded.index) * 0.2 / prev_aum.reindex(traded.index)).mean()
    aum = r["aum"].to_numpy()
    peak = np.maximum.accumulate(np.r_[capital, aum])
    rows.append({"account": capital, "days wanted": len(w), "sized to 0": f"{(w['contracts'] == 0).mean():.0%}",
                 "avg contracts": traded["contracts"].mean() if len(traded) else np.nan, "leverage": leverage,
                 "end balance": aum[-1], "CAGR": (aum[-1] / capital) ** (1 / years) - 1,
                 "max drawdown (closing)": ((peak[1:] - aum) / peak[1:]).max()})
sizing_by_account = pd.DataFrame(rows).set_index("account")
display(sizing_by_account.style.format({"avg contracts": "{:.1f}", "leverage": "{:.2f}", "end balance": "{:,.0f}", "CAGR": "{:.1%}", "max drawdown (closing)": "{:.1%}"}))

Below about R$ 5,000 the sizing never trades, and at R$ 5,000 it skips half the days and trades the rest at a higher leverage than it targets. At R$ 10,000 it is one contract most days and two on a few. From R$ 25,000 up the result is about the same whatever the account (it is the leverage target that sets it).

**One contract a day instead**, the least a trader can hold. Its P&L does not depend on the balance, so one run serves every account; what changes is how big a share of the account each swing is.

In [ ]:
rows = []
for capital in (1_000, 2_500, 5_000, 10_000, 25_000):
    balance = capital + per_contract.cumsum()
    peak = np.maximum.accumulate(np.r_[capital, balance.to_numpy()])
    rows.append({"account": capital, "end balance": balance.iloc[-1], "CAGR": (balance.iloc[-1] / capital) ** (1 / years) - 1,
                 "max drawdown (closing)": ((peak[1:] - balance.to_numpy()) / peak[1:]).max(),
                 "worst day, floating, % of account": -trough.min() / capital})
display(pd.DataFrame(rows).set_index("account").style.format({"end balance": "{:,.0f}", "CAGR": "{:.1%}", "max drawdown (closing)": "{:.1%}", "worst day, floating, % of account": "{:.0%}"}))

## 3. One contract per R$ X of balance

The rule: contracts = `max(1, floor(balance / X))`, so a second contract at 2X, a third at 3X. Each row is a different X. **A** starts the account at X (one contract at the start); **B** starts every row at R$ 10,000. A row that falls under R$ 1,000 (a contract's margin) stops trading. Costs are in. The balance moves day by day, closing P&L only.

In [ ]:
def simulate(per, start, X, floor=1000.0):
    balance, peak, max_dd, lowest, stopped = start, start, 0.0, start, None
    contracts = []
    for when, value in per.items():
        n = max(1, int(balance // X)) if balance >= floor else 0
        if n == 0 and stopped is None:
            stopped = when
        balance += n * value
        if value != 0:
            contracts.append(n)
        peak = max(peak, balance)
        max_dd = max(max_dd, (peak - balance) / peak)
        lowest = min(lowest, balance)
    return balance, max_dd, lowest, (np.mean(contracts) if contracts else 0.0), (max(contracts) if contracts else 0), stopped


def per_x_table(start_of):
    rows = []
    for X in range(1_000, 10_001, 1_000):
        start = start_of(X)
        end, dd, lowest, mean_c, max_c, stopped = simulate(per_contract, start, X)
        rows.append({"R$ per contract": X, "start": start, "end balance": end, "CAGR": (end / start) ** (1 / years) - 1,
                     "max drawdown (closing)": dd, "lowest balance": lowest, "contracts mean": mean_c, "contracts max": max_c,
                     "stopped": "" if stopped is None else f"{stopped:%Y-%m-%d}"})
    return pd.DataFrame(rows).set_index("R$ per contract")

fmt = {"start": "{:,.0f}", "end balance": "{:,.0f}", "CAGR": "{:.1%}", "max drawdown (closing)": "{:.1%}", "lowest balance": "{:,.0f}", "contracts mean": "{:.1f}"}
print("A. the account starts at X")
display(per_x_table(lambda X: X).style.format(fmt))
print("B. every row starts at R$ 10,000")
display(per_x_table(lambda X: 10_000).style.format(fmt))

With a small X the model compounds to nonsense (a R$ 10,000 account at R$ 1,000 a contract holds ten contracts on the first day, and thousands later). The rows to look at are the large X, where the number of contracts stays 1 to 4.

**Leverage this implies.** A contract is worth `0.2 x index`. At the last index of the window, the leverage a balance of exactly `n x X` carries with `n` contracts is `contract value / X`: the table below, and the margin it leaves before a 60% stop-out when a contract has just been added, measured in multiples of the worst floating drawdown of one contract in the run (`worst_dd`, which counts each day's floating loss). The model's figure comes from one-minute bars and is **lower** than what the tester measured tick by tick (R$ 2,271), so the last column is the more cautious one.

In [ ]:
INDEX_NOW = 171_000.0  # WIN$N on 2026-11-16 (the last day of the MT5 runs). The last close in this window is higher, before an overnight gap of -14.5% into 2026-10-26 in the continuous series that could not be checked.
contract_value = INDEX_NOW * 0.2
# the deepest FLOATING drawdown of one contract: the equity at the worst point of each day against the highest closing balance before it
closing = per_contract.cumsum()
before = np.r_[0.0, closing.to_numpy()[:-1]]
worst_dd = float((np.maximum.accumulate(before) - (before + trough.to_numpy())).max())
closing_dd = -float((closing - np.maximum(closing.cummax(), 0.0)).min())
print(f"a contract at an index of {INDEX_NOW:,.0f} is worth R$ {contract_value:,.0f}")
print(f"deepest drawdown of one contract: R$ {closing_dd:,.0f} on closing balances, R$ {worst_dd:,.0f} counting the floating loss of each day (the MT5 run showed R$ 2,271)")
MT5_FLOATING_DD = 2_271.0  # the equity drawdown MetaTrader 5 reported for one contract in the Strategy Tester (tick by tick)
rows = [{"R$ per contract": X, "leverage when a contract is added": contract_value / X,
         "worst drawdown (floating), share of the balance": worst_dd / X, "multiples of it to a 60% stop-out": 0.6 * X / worst_dd,
         "same, with MT5's R$ 2,271": 0.6 * X / MT5_FLOATING_DD}
        for X in range(2_000, 10_001, 1_000)]
display(pd.DataFrame(rows).set_index("R$ per contract").style.format(
    {"leverage when a contract is added": "{:.1f}x", "worst drawdown (floating), share of the balance": "{:.0%}", "multiples of it to a 60% stop-out": "{:.1f}x", "same, with MT5's R$ 2,271": "{:.1f}x"}))

## 4. Risk of ruin, with a broker that stops you out at a 60% loss

The daily P&L per contract is **resampled in blocks of 10 days** (a block keeps a losing streak together), 3 years a path, R$ 10,000 to start, contracts by the rule of section 3. Three versions of the edge, because the history is in sample:
- the **history**'s mean (as run);
- **half** of it (the history's mean cut in half, same spread);
- **none** (mean zero, same spread).

Two definitions of ruin:
- **Under R$ 1,000**: the balance cannot hold a contract.
- **Stop-out**: the equity (the balance plus the day's worst floating loss times the contracts) falls to **40% of the deposit**: a 60% loss of the account.

The stop-out is measured against the deposit. If the broker measures it against the balance of the day, it is at most reached with an X under about R$ 865 at these losses, so it never binds below.

In [ ]:
rng = np.random.default_rng(SEED)
v = per_contract.to_numpy(dtype=float)
t_day = trough.to_numpy(dtype=float)
n_days = len(v)
mean_edge = v.mean()


def resample(n_paths, horizon, block):
    n_blocks = -(-horizon // block)
    starts = rng.integers(0, n_days, size=(n_paths, n_blocks))
    return ((starts[:, :, None] + np.arange(block)[None, None, :]) % n_days).reshape(n_paths, -1)[:, :horizon]


def ruin(X, edge, horizon=756, n_paths=3000, start=10_000.0, block=10, floor=1000.0, stop_frac=0.4):
    idx = resample(n_paths, horizon, block)
    pnl = v[idx] - mean_edge * (1 - edge)  # edge 1: history, 0.5: half, 0: none
    floating = np.minimum(t_day[idx], pnl)
    balance = np.full(n_paths, float(start))
    peak = balance.copy()
    max_dd = np.zeros(n_paths)
    under, stopped, dead = (np.zeros(n_paths, dtype=bool) for _ in range(3))
    for s in range(horizon):
        n = np.where(~dead & (balance >= floor), np.maximum(1, np.floor(balance / X)), 0.0)
        equity = balance + n * floating[:, s]
        hit = (equity <= stop_frac * start) & ~dead
        stopped |= hit
        gone = (balance < floor) & ~dead
        under |= gone
        dead |= hit | gone
        balance = np.where(dead, balance, balance + n * pnl[:, s])
        peak = np.maximum(peak, balance)
        max_dd = np.maximum(max_dd, (peak - balance) / peak)
    return {"under R$ 1,000": under.mean(), "stopped out": stopped.mean(), "drawdown >= 30%": (max_dd >= 0.30).mean(),
            "drawdown >= 50%": (max_dd >= 0.50).mean(), "5th pct end": np.percentile(balance, 5), "median end": np.median(balance)}


XS = (1_000, 2_000, 3_000, 4_000, 5_000, 6_000, 7_000, 8_000, 9_000, 10_000)
ruin_tables = {}
for edge, label in ((1.0, "history"), (0.5, "half the edge"), (0.0, "no edge")):
    ruin_tables[label] = pd.DataFrame({X: ruin(X, edge) for X in XS}).T
    ruin_tables[label].index.name = "R$ per contract"
    print(f"{label}: R$ 10,000, 3 years, 3,000 paths")
    display(ruin_tables[label].style.format({"under R$ 1,000": "{:.1%}", "stopped out": "{:.1%}", "drawdown >= 30%": "{:.1%}",
                                              "drawdown >= 50%": "{:.1%}", "5th pct end": "{:,.0f}", "median end": "{:,.0f}"}))

How to read it: with the history's edge only the smallest X is in danger; with half the edge the risk moves up to X of R$ 2,000 to R$ 3,000; with none it reaches X of R$ 4,000 and a floor appears at about 0.2% to 0.5% for every X, which is the cost of trading one contract at all. **These are lower bounds**: a daily series from one run, blocks of 10 days (a regime of months, like 2022's, is not in it), and a spike inside a minute is not in the floating loss.

## 5. When to add the second contract, and how long it takes

At X = R$ 10,000 the second contract comes at R$ 20,000. One contract from R$ 10,000, the **first time the balance reaches R$ 20,000**, for a few monthly deposits (a deposit every 21 trading days), with the history's edge and with half of it.

In [ ]:
def first_time_at(target, edge, monthly, horizon=1260, n_paths=3000, start=10_000.0, block=10):
    idx = resample(n_paths, horizon, block)
    pnl = v[idx] - mean_edge * (1 - edge)
    balance = np.full(n_paths, float(start))
    when = np.full(n_paths, np.nan)
    for s in range(horizon):
        balance = balance + pnl[:, s]
        if monthly and s % 21 == 20:
            balance = balance + monthly
        hit = (balance >= target) & np.isnan(when)
        when[hit] = (s + 1) / 252
    return when

rows = []
for edge, label in ((1.0, "history"), (0.5, "half the edge")):
    for monthly in (0, 250, 500, 1000):
        when = first_time_at(20_000, edge, monthly)
        reached = ~np.isnan(when)
        rows.append({"edge": label, "monthly deposit": monthly, "reached in 3 years": np.mean(when <= 3), "reached in 5 years": reached.mean(),
                     "median years (if over half reach it)": np.nanmedian(when) if reached.mean() > 0.5 else np.nan})
display(pd.DataFrame(rows).set_index(["edge", "monthly deposit"]).style.format(
    {"reached in 3 years": "{:.0%}", "reached in 5 years": "{:.0%}", "median years (if over half reach it)": "{:.1f}"}))
print(f"in the history itself, one contract from R$ 10,000 ends at R$ {10_000 + per_contract.sum():,.0f} after {years:.1f} years")

**One way to run it** (a rule, not a result): trade one contract; look at the balance at each month-end; add the second at the first month-end at or above R$ 20,000 (the third at R$ 30,000); take one off only when the balance falls well under the threshold (about 10%), so the rule does not flip at the line. The buffer is a guess and is not simulated.

## 6. Head to head with the CDI, with a monthly deposit

Both start with R$ 10,000 invested at the close of the first day (2021-12-01) and receive the same deposit at the **close of the last trading day of each month**. The strategy account trades `max(1, floor(balance / R$ 10,000))` contracts, reset at each month-end after the deposit, with the daily P&L per contract of section 1 (costs in); its idle money earns nothing. The CDI account compounds the daily CDI on the whole balance, with no taxes. A last row is added at the end of the window (2026-10-16), without a deposit, because that month is not complete.

Extra column, **strategy + CDI**: the same trades, but the balance also earns the CDI each day, as if the broker accepted bonds as collateral. It is not what a plain account does; it shows what the futures P&L adds on top of the CDI. Whether a broker allows it is for you to check.

Then: the first 24 months one by one, each deposit level; then every 6 months; then a summary.

In [ ]:
cal = cdi[(cdi.index > bars.index[0].normalize()) & (cdi.index <= END)].index
daily_pnl = per_contract.reindex(cal).fillna(0.0).to_numpy()
daily_cdi = cdi.reindex(cal).to_numpy() / 100.0
month_last = pd.Series(cal, index=cal).groupby([cal.year, cal.month]).last()
complete = month_last[month_last < cdi.index.max()]  # a month whose last trading day the CDI file reaches
month_ends = set(complete.to_numpy())
X_CONTRACT = 10_000.0
START = 10_000.0


def head_to_head(deposit, start=START, x=X_CONTRACT):
    strategy = cdi_only = overlay = start
    contracts = 1
    rows, month = [], 0
    for i, when in enumerate(cal):
        strategy += contracts * daily_pnl[i]
        cdi_only *= 1 + daily_cdi[i]
        overlay = overlay * (1 + daily_cdi[i]) + contracts * daily_pnl[i]
        last = i == len(cal) - 1
        if when in month_ends or last:
            if when in month_ends:
                month += 1
                strategy += deposit
                cdi_only += deposit
                overlay += deposit
                contracts = max(1, int(strategy // x))
            rows.append((month if when in month_ends else month + 1, when, start + deposit * month,
                         strategy, cdi_only, overlay, contracts))
    out = pd.DataFrame(rows, columns=["month", "date", "paid in", "strategy", "CDI", "strategy + CDI", "contracts"])
    out["strategy - CDI"] = out["strategy"] - out["CDI"]
    return out

# checks: with no deposit and one contract, the strategy account is the start plus the cumulative P&L, and the CDI account is the
# benchmark module's factor
zero = head_to_head(0)
assert abs(zero["CDI"].iloc[-1] / START - compare_with_cdi(0.0, years, cdi, bars.index).factor) < 1e-9
assert zero["contracts"].max() == 1 and abs(zero["strategy"].iloc[-1] - (START + daily_pnl.sum())) < 1e-6
print("checks passed; complete months:", len(complete), "(2021-12 to " + f"{complete.iloc[-1]:%Y-%m})")

In [ ]:
DEPOSITS = (250, 500, 750, 1_000, 1_250)
runs = {d: head_to_head(d) for d in DEPOSITS}
money = {"paid in": "{:,.0f}", "strategy": "{:,.0f}", "CDI": "{:,.0f}", "strategy + CDI": "{:,.0f}", "strategy - CDI": "{:+,.0f}"}
for d, table in runs.items():
    print(f"\nMonthly deposit R$ {d:,}: the first 24 months")
    display(table.head(24).set_index("month").drop(columns="date").style.format(money))

### Every 6 months, then the end of the window

The 24-month table above is the head to head; this one continues it at months 30, 36, 42, 48, 54 and at the end of the window. `strategy - CDI` is the strategy account minus the CDI account, in reais; the percentage is that over the CDI account.

In [ ]:
marks = [30, 36, 42, 48, 54]
rows = []
for d, table in runs.items():
    pick = table[table["month"].isin(marks)].copy()
    pick = pd.concat([pick, table.iloc[[-1]]])
    pick["deposit"] = d
    rows.append(pick)
six = pd.concat(rows)
six["strategy vs CDI"] = six["strategy - CDI"] / six["CDI"]
six["date"] = six["date"].dt.strftime("%Y-%m-%d")
six_view = six.set_index(["deposit", "month"])[["date", "paid in", "strategy", "CDI", "strategy - CDI", "strategy vs CDI", "contracts", "strategy + CDI"]]
display(six_view.style.format({**money, "strategy vs CDI": "{:+.1%}"}))

### Summary by deposit

`months ahead` counts the month-ends (of the 58 complete ones plus the end of the window) where the strategy account is above the CDI account. `worst gap` is the largest amount the strategy account was below the CDI account at a month-end; `lowest strategy / paid in` is the lowest ratio of the strategy account to the money paid in. The strategy account is also measured by how far it fell from a high, taking each deposit out of the series (so a deposit does not hide a fall).

In [ ]:
def deposit_adjusted_drawdown(table, deposit):
    """Max drawdown of the strategy account in month-end steps, with the deposits taken out of each step."""
    balance = table["strategy"].to_numpy()
    paid = np.where(table["month"].to_numpy() <= len(complete), deposit, 0.0)  # the last row (a partial month) has no deposit
    before = np.r_[START, balance[:-1]]
    index = np.cumprod((balance - paid) / before)  # the return of each step: what the money made, the deposit not counted
    index = np.r_[1.0, index]
    return float((1 - index / np.maximum.accumulate(index)).max())

rows = []
for d, table in runs.items():
    rows.append({"deposit": d, "paid in": table["paid in"].iloc[-1], "strategy": table["strategy"].iloc[-1], "CDI": table["CDI"].iloc[-1],
                 "strategy - CDI": table["strategy - CDI"].iloc[-1], "months ahead": f"{(table['strategy'] > table['CDI']).sum()} of {len(table)}",
                 "worst gap": table["strategy - CDI"].min(), "lowest strategy / paid in": (table["strategy"] / table["paid in"]).min(),
                 "contracts at the end": table["contracts"].iloc[-1], "drawdown, deposits out": deposit_adjusted_drawdown(table, d),
                 "strategy + CDI": table["strategy + CDI"].iloc[-1]})
summary = pd.DataFrame(rows).set_index("deposit")
display(summary.style.format({"paid in": "{:,.0f}", "strategy": "{:,.0f}", "CDI": "{:,.0f}", "strategy - CDI": "{:+,.0f}", "worst gap": "{:+,.0f}",
                              "lowest strategy / paid in": "{:.3f}", "drawdown, deposits out": "{:.1%}", "strategy + CDI": "{:,.0f}"}))

## What to take from this, and what not to

- **The sizing is one dial.** Across accounts and across the rules of this notebook, the return and the drawdown move together. The choice is how much drawdown you can live with, and the broker's stop-out sets the floor on how much room to leave.
- **On a small account the rounding to whole contracts is the real constraint**, not the optimizer: below about R$ 5,000 the paper's sizing does not trade; one contract is the least there is and it is already a leverage of two to three or more.
- **The deposits drive the second contract more than the strategy does.** At R$ 10,000 a contract, the rule adds one at R$ 20,000, which profits alone reach in about four years (a little over half of the paths in five), and which deposits reach in under two years.
- **Against the CDI** the strategy account is ahead at the end for every deposit in the table, but it was **behind the CDI in many of the month-ends**, and its contracts grow with every deposit, so the account carries more leverage each year. The comparison is on the history the rule was chosen on; none of it is evidence the edge continues. Part of the reason a CDI account is hard to beat early is that the strategy's idle money earns nothing: the `strategy + CDI` column shows how much that is worth.
- **Not modelled:** the broker's real margin and stop-out rules, intraday spikes inside a minute, fill size effects at many contracts, taxes, and the time the money sits waiting between deposits.
